# Momentum Trading

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from trading_algos import datasets as tad
from trading_algos import utils as tau
from trading_algos.utils import head_tail as ht

2026-07-04 14:02:59.067 | INFO     | trading_algos.config:<module>:11 - PROJ_ROOT path is: /home/jamie/code/JamieW365/trading_algos


## Load Data

In [2]:
# Load S&P500 Surivors Dataset
df_stocks = tad.load_data()['Close']
ht(df_stocks)

Ticker,A,AA,AAL,AAP,AAPL,ABBV,ABNB,ABS,ABT,ACGL,...,XOM,XRAY,XRX,XYL,XYZ,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
1962-01-02,NaN,1.473408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.088754,NaN,0.635471,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-03,NaN,1.495946,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.090072,NaN,0.647386,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-04,NaN,1.495946,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.090292,NaN,0.643414,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-12,111.629997,NaN,NaN,NaN,255.759995,225.369995,127.699997,NaN,108.139999,94.220001,...,153.529999,NaN,NaN,120.019997,59.900002,158.460007,92.589996,204.050003,NaN,115.459999
2026-03-13,111.510002,NaN,NaN,NaN,250.119995,219.679993,126.300003,NaN,108.029999,93.470001,...,156.119995,NaN,NaN,119.879997,59.790001,160.399994,93.199997,202.720001,NaN,115.620003
2026-03-16,111.830002,NaN,NaN,NaN,252.820007,221.449997,128.320007,NaN,109.949997,93.699997,...,157.229996,NaN,NaN,121.070000,59.849998,161.779999,93.300003,203.970001,NaN,118.150002


## Strategy

#### Calculate Monthly and 12 Month Rolling Returns

In [8]:
# Daily Returns
ret = df_stocks.pct_change(fill_method=None).dropna(how='all') + 1
ht(ret)

Ticker,A,AA,AAL,AAP,AAPL,ABBV,ABNB,ABS,ABT,ACGL,...,XOM,XRAY,XRX,XYL,XYZ,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
1962-01-03,NaN,1.015296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.014852,NaN,1.018750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-04,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.002439,NaN,0.993865,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-05,NaN,0.998116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.978102,NaN,0.975309,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-12,0.967163,NaN,NaN,NaN,0.980637,0.989854,0.957343,NaN,0.980862,0.993777,...,1.012864,NaN,NaN,0.980956,0.926814,1.011877,0.987837,0.954977,NaN,0.963853
2026-03-13,0.998925,NaN,NaN,NaN,0.977948,0.974753,0.989037,NaN,0.998983,0.992040,...,1.016870,NaN,NaN,0.998834,0.998164,1.012243,1.006588,0.993482,NaN,1.001386
2026-03-16,1.002870,NaN,NaN,NaN,1.010795,1.008057,1.015994,NaN,1.017773,1.002461,...,1.007110,NaN,NaN,1.009927,1.001003,1.008604,1.001073,1.006166,NaN,1.021882


In [16]:
# Monthly Returns
# These are the returns would've seen had we invested for the month
mth_ret = ret.resample('ME').prod(min_count=1)
ht(mth_ret)

Ticker,A,AA,AAL,AAP,AAPL,ABBV,ABNB,ABS,ABT,ACGL,...,XOM,XRAY,XRX,XYL,XYZ,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
1962-01-31,NaN,0.913959,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.049505,NaN,0.918750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962-02-28,NaN,1.017683,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.056361,NaN,0.993198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962-03-31,NaN,1.055785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.986455,NaN,1.055504,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-01-31,0.985443,NaN,NaN,NaN,0.954462,0.983869,0.953212,NaN,0.876795,1.001251,...,1.175004,NaN,NaN,1.012410,0.928407,1.027895,0.968305,0.967713,NaN,0.996288
2026-02-28,0.906836,NaN,NaN,NaN,1.019066,1.040671,1.044369,NaN,1.064501,1.042795,...,1.085689,NaN,NaN,0.942912,1.054112,1.086399,1.130585,0.953102,NaN,1.050313
2026-03-31,0.921322,NaN,NaN,NaN,0.956999,0.954197,0.949745,NaN,0.944994,0.935597,...,1.031016,NaN,NaN,0.934471,0.939560,0.962060,0.947785,0.910743,NaN,0.901220


In [28]:
# Rolling 12 Month Returns
# At the beginning of each month the top 5 performing stocks will be
# Selected to be invested in for the next month
rolling_ret = mth_ret.rolling(12).apply(np.prod)
rolling_ret = rolling_ret.dropna(how='all', axis=0).dropna(how='all', axis=1)
ht(rolling_ret)

Ticker,A,AA,AAL,AAP,AAPL,ABBV,ABNB,ABT,ACGL,ACN,...,XEL,XOM,XRAY,XRX,XYL,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
1962-12-31,NaN,0.853254,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.234505,NaN,0.994766,NaN,NaN,NaN,NaN,NaN,NaN
1963-01-31,NaN,0.989125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.183688,NaN,1.040865,NaN,NaN,NaN,NaN,NaN,NaN
1963-02-28,NaN,0.922223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.127972,NaN,1.036808,NaN,NaN,NaN,NaN,NaN,NaN
2026-01-31,0.890493,NaN,NaN,NaN,1.104464,1.253350,0.986277,0.870405,1.031911,0.691707,...,1.168781,1.372176,NaN,NaN,1.124778,1.214639,0.803161,0.599531,NaN,0.740995
2026-02-28,0.956513,NaN,NaN,NaN,1.097136,1.147517,0.972924,0.858874,1.077925,0.604890,...,1.193902,1.416867,NaN,NaN,1.001956,1.096090,0.952956,0.710871,NaN,0.795308
2026-03-31,0.963671,NaN,NaN,NaN,1.143123,1.092396,1.074167,0.844426,0.974215,0.645110,...,1.189604,1.367462,NaN,NaN,1.025884,1.047870,0.830714,0.721864,NaN,0.728023


#### Identifying Stocks for Investing 

In [44]:
# We can identify the top 5 performing stocks for the previous 12 years
top_5 = rolling_ret.iloc[0].nlargest(5)
top_5

Ticker
CVX    1.237078
XOM    1.234505
LMT    1.209752
DTE    1.077709
ED     1.076103
Name: 1962-12-31 00:00:00, dtype: float64

In [55]:
# We know the date at which the previous 12 months are rolled up to
top_5.name

Timestamp('1962-12-31 00:00:00')

In [56]:
# We then know the 5 stocks that we will invest in for the following
# month
top_5.index

Index(['CVX', 'XOM', 'LMT', 'DTE', 'ED'], dtype='object', name='Ticker')

In [58]:
# We know the returns that we'd have seen if we had remained invested
# for the following month
mth_invested = mth_ret.loc[top_5.name:, top_5.index].iloc[1]
mth_invested

Ticker
CVX    1.031746
XOM    1.006303
LMT    0.953703
DTE    1.103174
ED     1.040247
Name: 1963-01-31 00:00:00, dtype: float64

In [59]:
# And we know what our overall monthly return would have been given an
# equal weighted portfolio
mth_invested.mean()

np.float64(1.0270347463017715)

#### Putting it all together

In [82]:
# Monthly strategy returns
inv_date = []
inv_ret = []


for date in rolling_ret.index[:-1]:

    top_5 = rolling_ret.loc[date].nlargest(5)

    mth_invested = mth_ret.loc[date:, top_5.index].iloc[1]

    inv_date.append(date)
    inv_ret.append(mth_invested.mean())



In [83]:
sp500_close = tad.load_data(local=False, tickers='^GSPC')['Close']
ht(sp500_close)

[*********************100%***********************]  1 of 1 completed


Ticker,^GSPC
Date,
1927-12-30,17.660000
1928-01-03,17.760000
1928-01-04,17.719999
2026-06-30,7499.359863
2026-07-01,7483.229980
2026-07-02,7483.240234


In [84]:
sp500_ret = sp500_close.pct_change().dropna() + 1
ht(sp500_ret)

Ticker,^GSPC
Date,
1928-01-03,1.005663
1928-01-04,0.997748
1928-01-05,0.990406
2026-06-30,1.007920
2026-07-01,0.997849
2026-07-02,1.000001


In [85]:
sp500_mth_ret = sp500_ret.resample('ME').prod()
ht(sp500_mth_ret)

Ticker,^GSPC
Date,
1928-01-31,0.994904
1928-02-29,0.982356
1928-03-31,1.117034
2026-05-31,1.051470
2026-06-30,0.989354
2026-07-31,0.997851


In [88]:
df_final = pd.DataFrame({'Date':inv_date, 'Strategy Returns':inv_ret}).set_index('Date')
ht(df_final)

,Strategy Returns
Date,
1962-12-31,1.027035
1963-01-31,0.963308
1963-02-28,1.038943
2025-12-31,1.293471
2026-01-31,1.054206
2026-02-28,0.972284


In [89]:
df_final['S&P500 Returns'] = sp500_mth_ret
ht(df_final)

,Strategy Returns,S&P500 Returns
Date,,
1962-12-31,1.027035,1.013492
1963-01-31,0.963308,1.049128
1963-02-28,1.038943,0.971148
2025-12-31,1.293471,0.999476
2026-01-31,1.054206,1.013663
2026-02-28,0.972284,0.991332


In [97]:
df_final.iloc[-10*12:].cumprod()

,Strategy Returns,S&P500 Returns
Date,,
2016-03-31,1.024465,1.065991
2016-04-30,1.093769,1.068869
2016-05-31,1.108414,1.085249
2016-06-30,1.178032,1.086237
2016-07-31,1.153889,1.124918
...,...,...
2025-10-31,22.309468,3.540055
2025-11-30,23.097621,3.544656
2025-12-31,29.876114,3.542798
